# gemma2asr：喺 Colab train reader

流程：**抽 feature（行一次凍結 encoder）→ 整字典 → train reader → 轉錄試聽**。

- 所有嘢（模型、feature、checkpoint）都放喺 Google Drive 嘅 `MyDrive/gemma2asr/`，斷線之後可以續做。
- 免費 T4 都得；L4／A100 會快好多（支援 bf16）。
- Runtime → Change runtime type → 揀 GPU。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 掛 Drive、攞代碼

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/gemma2asr'
os.makedirs(DRIVE, exist_ok=True)
os.environ['G2ASR_MODELS'] = f'{DRIVE}/models'  # 模型 cache 放 Drive，下次唔使再落載

# 方法 A：GitHub private repo。喺 Colab 左邊 🔑 Secrets 加 GH_TOKEN（fine-grained token，Contents: Read-only），開 Notebook access
REPO_URL = 'https://github.com/kuchris/gemma2asr.git'
# 方法 B：將 gemma2asr 資料夾 zip 咗放喺 MyDrive/gemma2asr/code.zip

%cd /content
if REPO_URL:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    auth_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
    !rm -rf gemma2asr && git clone -q {auth_url} gemma2asr && git -C gemma2asr remote set-url origin {REPO_URL}
else:
    !rm -rf gemma2asr && mkdir gemma2asr && unzip -q -o $DRIVE/code.zip -d gemma2asr
    # zip 入面如果多咗一層資料夾，就拉返出嚟
    inner = [d for d in os.listdir('gemma2asr') if os.path.isdir(f'gemma2asr/{d}/g2asr')]
    if inner:
        !mv gemma2asr/{inner[0]}/* gemma2asr/ && rm -rf gemma2asr/{inner[0]}
%cd /content/gemma2asr
!ls

In [ ]:
!pip install -q -U "transformers>=5.19.0" accelerate opencc-python-reimplemented soundfile librosa pyarrow huggingface_hub pillow

## 2.（可選）Spike 0：確認 audio 同文字喺同一個空間

本機已經做過：成段檢索 11/12，1 秒窗口 top-5 命中 97%（隨機 45%）。

In [ ]:
!python scripts/spike0.py

## 3. 抽 feature（最花時間，存喺 Drive，斷咗再行同一格會續做）

每小時音頻大約 92MB。可以先用 `--max-hours` 試細啲。

In [ ]:
FEATS = f'{DRIVE}/feats'
!python scripts/extract.py --dataset daily   --out $FEATS/daily
!python scripts/extract.py --dataset mixed   --out $FEATS/mixed
!python scripts/extract.py --dataset wordshk --out $FEATS/wordshk --max-hours 60

In [ ]:
# 由 Drive 抄去本機 SSD，train 起嚟快好多
!mkdir -p /content/feats && cp -r $FEATS/* /content/feats/ && du -sh /content/feats/*

## 4. 整字典（每個字嘅 text embedding）

In [ ]:
!python scripts/build_vocab.py /content/feats/daily /content/feats/mixed /content/feats/wordshk --out $DRIVE/vocab.pt --min-count 2

## 5. Train reader

- `--head frozen`：字典 embedding 固定（原本方法）
- `--head learned`：普通 CTC 分類頭（對照組），睇吓 EmbeddingGemma 嘅文字空間有冇幫到手
- 斷咗加 `--resume` 續 train

In [ ]:
!python scripts/train.py /content/feats/daily /content/feats/mixed /content/feats/wordshk     --vocab $DRIVE/vocab.pt --out $DRIVE/runs/frozen --head frozen --epochs 10

In [ ]:
# （可選）對照組
!python scripts/train.py /content/feats/daily /content/feats/mixed /content/feats/wordshk     --vocab $DRIVE/vocab.pt --out $DRIVE/runs/learned --head learned --epochs 10

## 6. 睇結果

In [ ]:
import json, matplotlib.pyplot as plt
for name in ['frozen', 'learned']:
    path = f'{DRIVE}/runs/{name}/log.jsonl'
    if os.path.exists(path):
        rows = [json.loads(l) for l in open(path)]
        plt.plot([r['step'] for r in rows], [r['cer'] for r in rows], label=name)
plt.ylabel('val CER'); plt.xlabel('step'); plt.legend(); plt.show()

In [ ]:
!python scripts/transcribe.py $DRIVE/runs/frozen/best.pt samples/*.wav

下載 `MyDrive/gemma2asr/runs/frozen/best.pt` 返本機，就可以用 `scripts/transcribe.py` 喺你部機試。